# 12. Qwen-Image-2.1で画像を生成する

このノートは、Google ColabのGPUでQwen-Image-2.1を動かし、**下書き → 本番**の2段階で画像を生成する実行例です。

## このノートで行うこと

1. GPU・VRAM・空きディスクを確認する
2. Diffusersと関連ライブラリをインストールする
3. モデルを読み込み、同じseedで下書きと本番を生成する
4. 完成したPNGをブラウザーから保存する

## 前提と注意

- L4またはA100を推奨します。T4ではメモリ不足や非常に長い実行時間になる可能性があります。
- 初回はモデルのダウンロードに時間とディスク容量が必要です。
- インストール後にランタイムの再起動が必要です。再起動後はGPU確認セルから続けます。
- 既定の本番解像度は高めです。L4で不安定な場合は設定セルの解像度を1280帯へ下げてください。
- パッケージの最新版でAPIが変わる場合があります。エラー時はリポジトリのIssueとノート内のバージョン指定を確認してください。

詳しい位置づけと、11・13番との違いは [追加ノート11〜13のガイド](../docs/07-notebooks-11-13.md) を参照してください。

---



# Qwen-Image-2.1 画像生成（Colab Pro）
**上から順に実行してください。** 最初に［ランタイム → ランタイムのタイプを変更］で GPU（可能なら L4 / A100）を選びます。モデルの初回ダウンロードには時間とディスク容量が必要です。生成後の PNG はブラウザーのダウンロード先へ保存します。Windows の標準設定では「ダウンロード」フォルダです。

モデル: https://huggingface.co/Qwen/Qwen-Image-2.1

## 画質を上げる手順
1. **下書き**: 1024×1024、25 step、CFG 1。構図だけ見る
2. **本番**: 同じ seed で 40 step。解像度を一段上げる（1280 帯、余裕があれば 2048）
3. **精度**: L4 は DiT もテキストエンコーダも int8 でよい。A100 は bf16 に上げると線が締まる。4bit や弱い GGUF は「動く」けれどきれいさは落ちる
4. **VAE**: Qwen-Image-2.1 純正の VAE（bf16）を使う。別モデルの VAE を流用するとにじむことがある
5. **ネガティブ**: CFG 1 のままだとネガティブはほぼ効かない。除外したいことは本文に書く。ネガティブを使うときだけ true_cfg_scale=2 程度（それ以上は濁りやすい）

※ 解像度を変えると同じ seed でも初期ノイズの形が変わるため、本番で構図が少し動くことがあります。

In [ ]:
import os
# VRAM の断片化を減らす（CUDA を初期化する前に設定する必要があるので最初のセルに置く）
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch, shutil
if not torch.cuda.is_available():
    raise RuntimeError('GPU がありません。ランタイムのタイプを GPU に変更してください。')
gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / 2**30
print(f'GPU: {gpu.name} / VRAM: {vram_gb:.1f} GiB / BF16: {torch.cuda.is_bf16_supported()}')
print(f'空きディスク: {shutil.disk_usage("/content").free / 2**30:.1f} GiB')
if not torch.cuda.is_bf16_supported():
    raise RuntimeError('このノートブックは BF16 対応 GPU（L4 / A100 など）を対象とします。GPU を変更してください。')
if shutil.disk_usage('/content').free < 45 * 2**30:
    print('警告: 空きディスクが 45 GiB 未満です。モデルのダウンロードに失敗する可能性があります。')

In [ ]:
%pip -q install -U git+https://github.com/huggingface/diffusers 'transformers>=5.17,<5.18' accelerate safetensors pillow torchao

インストールが完了したら［ランタイム → セッションを再起動］し、最初の GPU 確認セルとモデル読み込みセルを実行してください。インストールセルの再実行は不要です（torchao を追加したので、この版で初めて実行するときだけインストールセルを 1 回実行してください）。

そのあと **設定セル → 下書きセル → 本番セル** の順に進めます。

In [ ]:
import gc
import torch
from diffusers import QwenImage21Pipeline
try:
    from diffusers import AutoencoderKLQwenImage21
except ImportError:
    from diffusers.models.autoencoders.autoencoder_kl_qwenimage21 import AutoencoderKLQwenImage21

MODEL_ID = 'Qwen/Qwen-Image-2.1'

# 読み込み済みのモデルがあれば解放してから読み直す（二重に載って VRAM 不足になるのを防ぐ）
if 'pipe' in globals():
    del pipe
for _name in ('vae', 'draft_image', 'final_image'):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

# 精度: auto なら VRAM 38 GiB 以上（A100 など）は bf16、L4 は DiT だけ int8
# テキストエンコーダは bf16 のまま。torchao の int8 にすると CPU⇔GPU の入れ替えが一部しか動かず、
# VRAM 不足やデバイス不一致エラーの原因になるため（プロンプトの読み取り精度も bf16 のほうが上）
# bf16 は線が締まる。4bit や弱い GGUF は「動く」けれど画質が落ちるので選択肢に入れていない
PRECISION = 'auto'  # @param ['auto', 'bf16', 'int8']

vram_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
precision = PRECISION if PRECISION != 'auto' else ('bf16' if vram_gib >= 38 else 'int8')

# VAE は Qwen-Image-2.1 純正を bf16 で明示的に読み込む（別モデルの VAE を流用するとにじむことがある）
vae = AutoencoderKLQwenImage21.from_pretrained(MODEL_ID, subfolder='vae', torch_dtype=torch.bfloat16)

quant_config = None
if precision == 'int8':
    # diffusers では torchao の int8 weight-only が ComfyUI の int8 相当
    try:
        from torchao.quantization import Int8WeightOnlyConfig
        from diffusers import PipelineQuantizationConfig, TorchAoConfig as DiffusersTorchAoConfig
        quant_config = PipelineQuantizationConfig(quant_mapping={
            'transformer': DiffusersTorchAoConfig(quant_type=Int8WeightOnlyConfig()),
        })
    except Exception as e:
        print(f'int8 の準備に失敗したため bf16 + CPU オフロードで続行します: {e}')
        precision = 'bf16'

pipe = QwenImage21Pipeline.from_pretrained(
    MODEL_ID, vae=vae, torch_dtype=torch.bfloat16, quantization_config=quant_config,
)

# 配置: 全部を GPU に常駐させると、生成の最後の VAE デコードで VRAM が足りなくなる（L4 の int8 で約 21.7 GiB 使用）
# そこで VRAM 70 GiB 未満では「使うモデルだけ GPU に載せる」方式にする。
# テキストエンコード → DiT → VAE の順に GPU へ入れ替わるので、デコード時は VAE 分の余裕ができる
if vram_gib >= 70:
    pipe.to('cuda')
    placement = 'GPU 常駐'
else:
    pipe.enable_model_cpu_offload()
    placement = 'モデル単位で GPU に入れ替え（enable_model_cpu_offload）'

print(f'精度: DiT = {precision} / テキストエンコーダ = bf16')
print(f'VAE: {type(pipe.vae).__name__} / {pipe.vae.dtype}')
print(f'配置: {placement}')

In [ ]:
import gc
import sys
import random
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from IPython.display import display
from google.colab import files

# VRAM 70 GiB 未満なのにモデルが GPU に常駐していたら、使うモデルだけ GPU に載せる方式へ切り替える
# （常駐のままだと最後の VAE デコードで VRAM 不足になる。モデル読み込みセルを再実行しなくても効く）
def ensure_offload():
    vram = torch.cuda.get_device_properties(0).total_memory / 2**30
    if vram < 70 and pipe.transformer.device.type == 'cuda':
        pipe.enable_model_cpu_offload()
        torch.cuda.empty_cache()
        print('モデル単位で GPU に入れ替える方式に切り替えました。')

ensure_offload()

# L4 など VRAM が少ない GPU で 2048 帯を出すときは VAE をタイル分割でデコードする（デコード時の VRAM 不足対策）
def prepare_vae(band):
    if not hasattr(pipe.vae, 'enable_tiling'):
        return
    if band >= 2048 and torch.cuda.get_device_properties(0).total_memory / 2**30 < 38:
        pipe.vae.enable_tiling()
    elif hasattr(pipe.vae, 'disable_tiling'):
        pipe.vae.disable_tiling()

# ===== 共通設定（下書き・本番で同じ値を使う） =====
prompt = '久保帯人のBLEACH公式イラスト風。千年血戦篇のアニメ塗り。鋭い輪郭、細い線、強い黒。成人女性の四楓院夜一、1人。20代後半。褐色肌。紫の長いポニーテール。金色の切れ長の目。高い頬骨。大人びた顔立ち。甘いのに抵抗する表情。汗。全裸。大きな胸。ピンクの乳輪。引き締まった腰。手は2本。指は5本。成人男性が1人。がっしりした体。短い黒髪。顔は横顔。手は2本。指は5本。夜一さんが上に跨り、男性器が女性器に挿入されている。ベッド。夜一さんがこちらを見ている。暗い寝室。高い解像度。線がはっきりしたセル画。同じ絵のまま。顔も部屋も体勢も変えない。夜一さんは男性の上に腰を落としている。男性器は女性器の中に入っている。結合部だけ描き直す。男性器が腹の前に出ない。根本まで挿入されている。夜一さんの腰が男性の腰に近い。'

# 除外したいことはここに書いて本文へ足す（CFG 1 ではネガティブプロンプトがほぼ効かない）
# 例: '文字、ロゴ、透かしは入れない。背景はシンプルにする。'
exclude_in_prompt = ''

ASPECT = '1:1'  # @param ['1:1', '4:3', '3:4', '3:2', '2:3', '16:9', '9:16']
seed = 42  # @param {type:'integer'}
# -1 にするとランダム。気に入った構図が出たらその seed を固定して本番へ

# ネガティブを使いたいときだけ True（true_cfg_scale=2 程度。それ以上は濁りやすい）
USE_NEGATIVE = False  # @param {type:'boolean'}
negative_prompt = ''
TRUE_CFG_SCALE = 2.0

if seed < 0:
    seed = random.randrange(2**31)
print(f'seed: {seed}')

# 公式の推奨比率（2048 帯）。band に合わせて縮小し 32 の倍数に丸める
ASPECT_RATIOS = {
    '1:1': (2048, 2048), '4:3': (2400, 1792), '3:4': (1792, 2400),
    '3:2': (2528, 1696), '2:3': (1696, 2528), '16:9': (2752, 1536), '9:16': (1536, 2752),
}

def size_for(band):
    w, h = ASPECT_RATIOS[ASPECT]
    scale = band / 2048
    return round(w * scale / 32) * 32, round(h * scale / 32) * 32

def generate(stage, band, steps, download):
    # 前回のエラーの traceback が VRAM を握ったままになるのを解放する
    sys.last_traceback = sys.last_value = sys.last_type = None
    gc.collect()
    torch.cuda.empty_cache()
    prepare_vae(band)
    width, height = size_for(band)
    full_prompt = prompt if not exclude_in_prompt.strip() else f'{prompt} {exclude_in_prompt}'
    kwargs = dict(prompt=full_prompt, width=width, height=height, num_inference_steps=steps)
    if USE_NEGATIVE and negative_prompt.strip():
        if TRUE_CFG_SCALE > 2.5:
            print('注意: true_cfg_scale が高めです。濁りやすくなります。')
        kwargs.update(negative_prompt=negative_prompt, true_cfg_scale=TRUE_CFG_SCALE)
        cfg = TRUE_CFG_SCALE
    else:
        kwargs.update(true_cfg_scale=1.0)
        cfg = 1.0
    print(f'[{stage}] {width}x{height} / {steps} step / CFG {cfg} / seed {seed}')
    for attempt in range(2):
        try:
            image = pipe(**kwargs, generator=torch.Generator('cuda').manual_seed(seed)).images[0]
            break
        except torch.OutOfMemoryError:
            torch.cuda.empty_cache()
            if attempt == 1:
                raise RuntimeError('VRAM 不足です。セッションを再起動し、GPU 確認セルから順に実行し直してください。')
            print('VRAM 不足のため、モデル入れ替え方式と VAE タイリングを有効にして再試行します。')
            ensure_offload()
            if hasattr(pipe.vae, 'enable_tiling'):
                pipe.vae.enable_tiling()
    stamp = datetime.now(ZoneInfo('Asia/Tokyo')).strftime('%Y%m%d_%H%M%S')
    output_path = Path('/content') / f'qwen_image_2_1_{stage}_{width}x{height}_{steps}step_seed{seed}_{stamp}.png'
    image.save(output_path)
    display(image)
    print(f'生成画像: {output_path} ({output_path.stat().st_size:,} bytes)')
    if download:
        files.download(str(output_path))
    return image

In [ ]:
# 1. 下書き: 1024 帯 / 25 step / CFG 1。構図だけ確認する
# 構図が気に入らなければ上の seed を変えて（または -1 で）設定セルとこのセルを再実行
DRAFT_DOWNLOAD = False  # @param {type:'boolean'}
draft_image = generate('draft', band=1024, steps=25, download=DRAFT_DOWNLOAD)

In [ ]:
# 2. 本番: 下書きと同じ seed で 40 step。解像度を一段上げる
# L4 は 1280 帯が安定。A100 など余裕があれば 2048（公式の推奨解像度）
FINAL_BAND = 2048  # @param [1280, 2048] {type:'raw'}
final_image = generate('final', band=FINAL_BAND, steps=40, download=True)

**使い方**: 構図探しは設定セルの `prompt` / `seed` / `ASPECT` を変えて、設定セル → 下書きセルを再実行。構図が決まったら本番セルを実行します（設定セルは再実行しなくて OK。seed がそのまま引き継がれます）。

本番の PNG はブラウザー経由で Windows のダウンロード先に保存されます。保存先を変更している場合はブラウザーの設定に従います。下書きも保存したいときは `DRAFT_DOWNLOAD` を True にします。